# Notebook 02 — Feature Engineering
Extract numeric features and character token sequences. Build and save train/val/test splits for both layers.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from feature_engineering.extractor import extract_features, to_vector, FEATURE_NAMES, INPUT_DIM
from feature_engineering.tokenizer import CharTokenizer
from feature_engineering.normalizer import FeatureNormalizer
from sklearn.model_selection import train_test_split

DATA_PROC  = Path("../data/processed")
SPLITS_DIR = Path("../data/splits")
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROC / "csic_parsed.csv")
print(f"Loaded {len(df)} records")
print(df["attack_class"].value_counts())

## 1. Extract numeric features for all records

In [ ]:
from tqdm import tqdm

feature_rows = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
    req = {
        "url":     str(row.get("url",    "")),
        "method":  str(row.get("method", "GET")),
        "headers": row.get("headers", {}),
        "body":    str(row.get("body",  "")),
    }
    feature_rows.append(extract_features(req))

df_feat = pd.DataFrame(feature_rows)
df_feat["label_id"]       = df["label_id"].values
df_feat["attack_class_id"] = df["attack_class_id"].values
print(df_feat.shape)
df_feat.head()

## 2. Visualise feature distributions

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for i, feat in enumerate(FEATURE_NAMES):
    ax = axes[i // 5][i % 5]
    for label_id, color in [(0, "steelblue"), (1, "coral")]:
        vals = df_feat[df_feat["label_id"] == label_id][feat].clip(
            df_feat[feat].quantile(0.01), df_feat[feat].quantile(0.99))
        ax.hist(vals, bins=30, alpha=0.5, color=color,
                label="normal" if label_id == 0 else "attack")
    ax.set_title(feat, fontsize=8)
    ax.legend(fontsize=6)
plt.tight_layout()
plt.savefig("../data/processed/02_feature_distributions.png", dpi=100)
plt.show()

## 3. Correlation heatmap

In [ ]:
corr = df_feat[FEATURE_NAMES].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".1f", cmap="coolwarm", center=0,
            annot_kws={"size": 6})
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.savefig("../data/processed/02_correlation.png", dpi=100)
plt.show()

## 4. Build Layer 2A splits (normal-only train, mixed test)

In [ ]:
X_all   = df_feat[FEATURE_NAMES].values.astype(np.float32)
y_all   = df_feat["label_id"].values.astype(int)

# normal-only pool for L2A training
X_normal = X_all[y_all == 0]
X_attack = X_all[y_all == 1]

X_n_train, X_n_temp = train_test_split(X_normal, test_size=0.30, random_state=42)
X_n_val,  X_n_test  = train_test_split(X_n_temp, test_size=0.50, random_state=42)

# mixed test: remaining normal + all attacks
X_test_l2a = np.vstack([X_n_test, X_attack])
y_test_l2a = np.array([0] * len(X_n_test) + [1] * len(X_attack), dtype=int)

# shuffle test set
idx = np.random.RandomState(42).permutation(len(X_test_l2a))
X_test_l2a, y_test_l2a = X_test_l2a[idx], y_test_l2a[idx]

np.save(SPLITS_DIR / "l2a_normal_train.npy", X_n_train)
np.save(SPLITS_DIR / "l2a_normal_val.npy",   X_n_val)
np.save(SPLITS_DIR / "l2a_test_X.npy",       X_test_l2a)
np.save(SPLITS_DIR / "l2a_test_y.npy",       y_test_l2a)

print(f"L2A splits:")
print(f"  train (normal): {X_n_train.shape}")
print(f"  val   (normal): {X_n_val.shape}")
print(f"  test  (mixed):  {X_test_l2a.shape}  attack rate={y_test_l2a.mean():.2%}")

## 5. Build Layer 2B splits (labeled, stratified, numeric + tokens)

In [ ]:
y_mc = df_feat["attack_class_id"].values.astype(int)

X_tr, X_temp, y_tr, y_temp = train_test_split(
    X_all, y_mc, test_size=0.30, stratify=y_mc, random_state=42)
X_v, X_te, y_v, y_te = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

np.save(SPLITS_DIR / "l2b_train_X_numeric.npy", X_tr)
np.save(SPLITS_DIR / "l2b_val_X_numeric.npy",   X_v)
np.save(SPLITS_DIR / "l2b_test_X_numeric.npy",  X_te)
np.save(SPLITS_DIR / "l2b_train_y.npy", y_tr)
np.save(SPLITS_DIR / "l2b_val_y.npy",   y_v)
np.save(SPLITS_DIR / "l2b_test_y.npy",  y_te)

print(f"L2B numeric splits:")
print(f"  train: {X_tr.shape}  classes: {dict(zip(*np.unique(y_tr, return_counts=True)))}")
print(f"  val:   {X_v.shape}")
print(f"  test:  {X_te.shape}")

## 6. Build character token sequences for CNN / GRU

In [ ]:
tok = CharTokenizer(max_len=512)

# rebuild request strings from df
def row_to_text(row):
    return (str(row.get("method","GET")) + " "
            + str(row.get("url","")) + " "
            + str(row.get("body","")))

texts = df.apply(row_to_text, axis=1).tolist()
print(f"Tokenising {len(texts)} requests...")

# split indices match numeric splits (same random state)
tr_idx   = np.where(np.isin(np.arange(len(df)),
    train_test_split(np.arange(len(df)), test_size=0.30, random_state=42)[0]))[0]

# re-use the same indices as the numeric split
from sklearn.model_selection import train_test_split as tts
idx_all = np.arange(len(df))
idx_tr, idx_temp = tts(idx_all, test_size=0.30, stratify=y_mc, random_state=42)
idx_v,  idx_te   = tts(idx_temp, test_size=0.50,
                       stratify=y_mc[idx_temp], random_state=42)

tok_texts = [texts[i] for i in range(len(texts))]
X_tok_all = tok.encode_batch(tok_texts)

np.save(SPLITS_DIR / "l2b_train_X_tokens.npy", X_tok_all[idx_tr])
np.save(SPLITS_DIR / "l2b_val_X_tokens.npy",   X_tok_all[idx_v])
np.save(SPLITS_DIR / "l2b_test_X_tokens.npy",  X_tok_all[idx_te])
print(f"Token splits saved. Shape per split: {X_tok_all[idx_tr].shape}")

## 7. Fit and save normalizer for L2A

In [ ]:
norm = FeatureNormalizer()
norm.fit(X_n_train)
norm.save("../exported_models/scaler_l2a.pkl")
print("Normalizer stats (first 5 features):")
stats = norm.stats()
for k in list(stats.keys())[:5]:
    print(f"  {k}: mean={stats[k]['mean']}  std={stats[k]['std']}")
print("Feature engineering complete.")